In [6]:
df_links = pd.read_csv("임시공휴일링크3_본문_크롤링결과.csv", encoding='cp949', sep='\t', on_bad_lines='skip')
df_links

,링크,본문내용,날짜
0,https://www.instagram.com/p/CDzvSCbh6Sm/,본문 없음,날짜 없음
1,https://www.instagram.com/p/CDzvSCbh6Sm/,본문 없음,날짜 없음
2,https://www.instagram.com/p/CDzvSCbh6Sm/,lagoonia998\n?\n팔로우\n경상북도 울진\nlagoonia998\n 수정...,2023-03-15T08:26:30.000Z
3,https://www.instagram.com/p/CSrpzblhHqi/,mk220125\n?\n팔로우\nmk220125\n 수정됨\n?\n189주\n.\n...,2021-08-18T08:18:55.000Z
4,https://www.instagram.com/p/CD-pDWDlDO7/,ksae74\n?\n팔로우\nksae74\n 241주\n아빠표 김치볶음밥.?\n장마...,2020-08-17T05:31:18.000Z
...,...,...,...
504,https://www.instagram.com/p/CD_K3OfFebA/,queen_nail6611\n?\n팔로우\n청도계곡\nqueen_nail6611\n...,2020-08-17T10:26:44.000Z
505,https://www.instagram.com/p/DA7S6gvSIKH/,insight_biz\n?\n팔로우\ninsight_biz\n 25주\n”어버이날·...,2024-10-10T02:15:11.000Z
506,https://www.instagram.com/p/CC45C7An6OL/,readyonnews\n?\n팔로우\nreadyonnews\n 수정됨\n?\n245...,2020-07-21T04:22:42.000Z
507,https://www.instagram.com/p/CD_UsXUB9RK/,banghyojin4\n?\n팔로우\n천성리버타운\nbanghyojin4\n 241...,2020-08-17T11:52:38.000Z


In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random
from pathlib import Path

USERNAME = '***REMOVED***'
PASSWORD = '***REMOVED***'

# 1. 크롬 설정
options = Options()
options.add_experimental_option("detach", True)
options.add_experimental_option("excludeSwitches", ["enable-logging"])
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

# 2. 로그인
driver.get('https://www.instagram.com/accounts/login/')
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "username")))
driver.find_element(By.NAME, 'username').send_keys(USERNAME)
driver.find_element(By.NAME, 'password').send_keys(PASSWORD)
driver.find_element(By.NAME, 'password').send_keys(Keys.ENTER)
print("✅ 로그인 완료")

# 3. 팝업 닫기
time.sleep(5)
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[text()='나중에 하기']"))
    ).click()
    print("✅ 팝업 닫기 완료")
except:
    print("ℹ️ 팝업 없음")

# 4. 링크 불러오기
df_links = pd.read_csv(r"C:\Users\kstat\Documents\크롤링\임시공휴일\임시공휴일_링크2.csv", encoding='cp949')
links = df_links[df_links.columns[0]].dropna().tolist()
print(f"🔗 총 링크 수: {len(links)}")

# 5. 본문 추출 함수
def extract_post_content(driver, url):
    try:
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(2)

        try:
            content_div = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, '//section/main/div/div[1]/div/div[2]'))
            )
            content = content_div.text
        except:
            content = "본문 없음"

        try:
            date = driver.find_element(By.TAG_NAME, 'time').get_attribute('datetime')
        except:
            date = "날짜 없음"

        return {"링크": url, "본문내용": content, "날짜": date}

    except Exception as e:
        print(f"❌ 실패: {url} ➡ {e}")
        return {"링크": url, "본문내용": "", "날짜": ""}

# 6. 결과 저장 경로 및 초기화
save_path = Path("임시공휴일링크4_본문_크롤링결과.csv")
columns = ["링크", "본문내용", "날짜"]

# 첫 실행 시 헤더 생성
if not save_path.exists():
    pd.DataFrame(columns=columns).to_csv(save_path, index=False, encoding='utf-8-sig')
    print("📄 새로운 CSV 생성 완료")

# 7. 반복 크롤링 및 실시간 저장
for idx, link in enumerate(links, 1):
    print(f"📥 ({idx}/{len(links)}) 크롤링 중: {link}")
    data = extract_post_content(driver, link)

    # 한 줄씩 바로 저장
    pd.DataFrame([data]).to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')

    time.sleep(random.uniform(2, 4))

print("✅ 크롤링 완료 및 모든 데이터 저장됨!")


✅ 로그인 완료
ℹ️ 팝업 없음
🔗 총 링크 수: 6388
📥 (1/6388) 크롤링 중: https://www.instagram.com/p/CD_UDJBJORI/
📥 (2/6388) 크롤링 중: https://www.instagram.com/p/CD9KERmAPrh/
📥 (3/6388) 크롤링 중: https://www.instagram.com/p/CDie1H4BBOF/
📥 (4/6388) 크롤링 중: https://www.instagram.com/p/CD31xxynnya/
📥 (5/6388) 크롤링 중: https://www.instagram.com/p/CD-sBZcl0Ym/
📥 (6/6388) 크롤링 중: https://www.instagram.com/p/CIo9g48lEFQ/
📥 (7/6388) 크롤링 중: https://www.instagram.com/p/CD-p6fBlDf5/
📥 (8/6388) 크롤링 중: https://www.instagram.com/p/Cwl0vYuxpc8/
📥 (9/6388) 크롤링 중: https://www.instagram.com/p/CSn6hSKHswA/
📥 (10/6388) 크롤링 중: https://www.instagram.com/p/CyABMIxy90d/
📥 (11/6388) 크롤링 중: https://www.instagram.com/p/DA5QfvqTEek/
📥 (12/6388) 크롤링 중: https://www.instagram.com/p/CieGabsP-mz/
📥 (13/6388) 크롤링 중: https://www.instagram.com/p/CyVSR1yBHK7/
📥 (14/6388) 크롤링 중: https://www.instagram.com/p/CD_HyZ9lXfJ/
📥 (15/6388) 크롤링 중: https://www.instagram.com/p/BioOx38l6af/
📥 (16/6388) 크롤링 중: https://www.instagram.com/p/CUlklVAv7M3/
📥 (17/6388) 크롤링 

PermissionError: [Errno 13] Permission denied: '임시공휴일링크4_본문_크롤링결과.csv'

In [15]:
import pandas as pd

# 파일 경로
csv_path = "최종(인스타아직아님).csv"
csv2_path = "임시공휴일_통합텍스트.csv"
xlsx_path = "합칠것.xlsx"

# 각각 불러오기
df_csv = pd.read_csv(csv_path, encoding='utf-8')
df_xlsx = pd.read_excel(xlsx_path)

# 컬럼명이 같은지 확인하고 정렬 맞추기
#df_xlsx.columns = df_csv.columns  # 열 이름 강제 일치 (순서 기준)

# 아래로 이어붙이기 (index 무시)
df_combined = pd.concat([df_csv, df_xlsx], ignore_index=True)

# 저장
combined_path = "최종_세로병합_완료.xlsx"
df_combined.to_excel(combined_path, index=False)


In [14]:
# 기존 text 컬럼 가져오기
text_existing = df_csv["text"].dropna().tolist()

# 새로 넣을 본문내용 가져오기
text_to_add = df_xlsx["본문내용"].dropna().tolist()

# 이어붙이기
merged_text = text_existing + text_to_add

# 데이터프레임으로 재구성
df_merged = pd.DataFrame({"text": merged_text})

# 저장
merged_text_path = "최종_text컬럼_이어붙이기.xlsx"
df_merged.to_excel(merged_text_path, index=False)

merged_text_path

'최종_text컬럼_이어붙이기.xlsx'

In [19]:
import pandas as pd

# 파일 불러오기
df1 = pd.read_excel("최종_text컬럼_이어붙이기.xlsx")
df2 = pd.read_csv("임시공휴일_통합텍스트.csv", encoding="utf-8")

# 'text' 컬럼 추출
text_existing = df1['text'].dropna().tolist()
text_to_add = df2['text'].dropna().tolist()

# 결합
merged_text = text_existing + text_to_add

# 새 데이터프레임 생성 및 저장
df_merged = pd.DataFrame({'text': merged_text})
df_merged.to_excel("결합된_텍스트.xlsx", index=False)


In [21]:
import pandas as pd

# 파일 불러오기
df_merged
df2 = pd.read_csv("임시공휴일링크4_본문_크롤링결과.csv", encoding="utf-8")
# 'text' 컬럼 추출
text_existing = df_merged['text'].dropna().tolist()
text_to_add = df2['본문내용'].dropna().tolist()

# 결합
merged_text = text_existing + text_to_add

# 새 데이터프레임 생성 및 저장
df_merged = pd.DataFrame({'text': merged_text})
df_merged.to_excel("결합된_텍스트.xlsx", index=False)